# Model initiation

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor
from tqdm import tqdm
device = 'cuda' if torch.cuda.is_available else 'cpu'
# ckpt = "google/siglip2-giant-opt-patch16-384"
ckpt = "google/siglip2-so400m-patch14-384"
model = AutoModel.from_pretrained(
    ckpt,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" # Install flash-attn before using this
).eval()
processor = AutoProcessor.from_pretrained(ckpt)

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

# Load local images

In [2]:
from glob import glob

img_paths = glob(r'D:\AIC_LITE\kfs\Keyframes_L01\keyframes\*\*.jpg')
img_codes = [f'{path.split('\\')[-2]}_{path.split('\\')[-1].split('.')[0]}' for path in img_paths]

# Encoding

In [4]:
from PIL import Image

NUM_BATCHES = 128
img_feats = {}
test = []
for i in tqdm(range(0, len(img_paths), NUM_BATCHES)):
    j = len(img_paths) if i + NUM_BATCHES > len(img_paths) else i + NUM_BATCHES
    codes = img_codes[i:j]
    raws = [Image.open(path) for path in img_paths[i:j]]
    preprocs = processor(images=raws, return_tensors='pt')
    
    with torch.no_grad():
        feats = model.get_image_features(**preprocs.to(device)).pooler_output
        feats = F.normalize(feats)

    img_feats.update({code: feat.cpu() for code, feat in zip(codes, feats)})

100%|██████████| 64/64 [06:36<00:00,  6.19s/it]


# Saving to individual samples

In [ ]:
import numpy as np

for code, feat in tqdm(list(img_feats.items())):
    np.save(f'../additional_data/feat_siglip2_large/{code}.npy', feat.float().numpy())

100%|██████████| 8115/8115 [00:05<00:00, 1376.65it/s]
